In [2]:
import earthaccess
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

# 1. Authenticate using the ~/.netrc we just created
auth = earthaccess.login()

# 2. Define your Gadi Scratch directory for storage (Home is too small!)
# Replace 'zr7147' with your NCI username if different
output_dir = "/g/data/k10/zr7147/GPM_IMERG_Data"
os.makedirs(output_dir, exist_ok=True)

print(f"Authenticated. Data will be saved to: {output_dir}")

Authenticated. Data will be saved to: /g/data/k10/zr7147/GPM_IMERG_Data


In [3]:
# Define ORCESTRA region: [Lon Min, Lat Min, Lon Max, Lat Max]
bbox = (-65, 5, -15, 20) 

# The campaign Date Range (YYYY-MM-DD)
date_range = ("2024-08-10", "2024-09-30")

# Search for GPM IMERG Final Run (Half-Hourly)
results = earthaccess.search_data(
    short_name="GPM_3IMERGHH",
    bounding_box=bbox,
    temporal=date_range
)

print(f"Found {len(results)} half-hourly files for these dates.")

# Download only the files we don't already have
downloaded_files = earthaccess.download(results, local_path=output_dir)

Found 2496 half-hourly files for these dates.


QUEUEING TASKS | :   0%|          | 0/2496 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2496 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2496 [00:00<?, ?it/s]

In [6]:
# Load all downloaded files into one dataset
# IMERG files usually have names like '3B-HHR.MS.MRG.3IMERG...'
file_pattern = os.path.join(output_dir, "*.HDF5") 
ds_gpm = xr.open_mfdataset(file_pattern, concat_dim='time', combine='nested', engine='h5netcdf', group='/Grid')

# Extract 'precipitation' (Calibrated Precipitation in mm/hr)
# Note: IMERG longitude is -180 to 180. We slice to our flight region.
precip = ds_gpm['precipitation'].sel(lon=slice(-65, -15), lat=slice(5, 20))

# Calculate a daily mean for the overview plot
daily_mean = precip.mean(dim='time')

/scratch/nf33/zr7147/tmp/ipykernel_2092433/1558644469.py:4: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_gpm = xr.open_mfdataset(file_pattern, concat_dim='time', combine='nested', engine='h5netcdf', group='/Grid')
